# PRT-DeepONet — Monod kinetics (concentration variant, full training pipeline)

Trains the **transient** concentration DeepONet for Monod reaction kinetics. Like the sorption
model it is **velocity-conditioned** (CVB), but the trunk carries an extra **normalized time**
channel so a single model predicts the whole time evolution.

**Branch CNN** ← `[mask, ux, uy]` (predicted velocity, z-scored).  
**Branch FNN** ← the parametric conditions `(Pe, Da)`.  
**Trunk** ← `(x, y)`, the normalized time `t`, and the **GDF** (inlet distance):  `trunk = [x, y, t, GDF]`.

The four training conditions are the corners `Pe ∈ {1, 10} × Da ∈ {7.4, 74}`. The released
checkpoint is `../parameters/Monod.pt`. Concentration is evaluated by **RMSE** over the porous ROI.

> **Note.** As in the reference repo, the trunk tensor `[x, y, t, GDF]`, the base mask channel,
> the `(Pe, Da)` branch input and the transient labels are read from the pre-built dataset caches
> `monod_{train,test}_dataset_trunk4.pt` (built once from the raw Monod snapshots). This notebook
> then adds the velocity-conditioning channels and trains the CVB model.


In [ ]:
# ====== 0. Imports ======
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
# ====== 1. Reproducibility & Paths ======
import os
nx, ny = 64, 148
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED, BATCH, LR, EPOCHS, PATIENCE = 42, 25, 1e-3, 500, 15

# All inputs live under one data root — set $PRT_DATA_ROOT or edit DATA_ROOT.
# Expected layout is documented in the README ("Data layout").
DATA_ROOT = os.environ.get('PRT_DATA_ROOT', '../../data')
MONNPZ    = f'{DATA_ROOT}/concentration/monod/masks'        # raw masks m{dom}.npz
CACHE     = f'{DATA_ROOT}/concentration/monod/trunk_cache'  # pre-built trunk4 datasets + meta
VELC_PATH = f'{DATA_ROOT}/predicted_velocity.npz'           # predicted velocity (ux,uy), λ=10
MODEL_OUT = '../parameters/Monod.pt'                           # trained output
# four training conditions: (name, Pe, Da)
groups = [('P1D7.4',1,7.4), ('P1D74',1,74.0), ('P10D7.4',10,7.4), ('P10D74',10,74.0)]
def pe_of(g): return groups[g][1]


In [ ]:
# ====== 2. Model Definition (paper-style names) ======
class CustomCNN(nn.Module):
    def __init__(s, in_channels, out_dim, num_blocks=5):
        super().__init__(); ch=[in_channels,16,32,64,128,256][:num_blocks+1]; L=[]
        for i in range(num_blocks): L+=[nn.Conv2d(ch[i],ch[i+1],3,1,1), nn.SiLU(), nn.AvgPool2d(2)]
        s.features=nn.Sequential(*L); h,w=64,148
        for _ in range(num_blocks): h//=2; w//=2
        s.fc=nn.Linear(ch[num_blocks]*h*w, out_dim)
    def forward(s, x): x=s.features(x); return s.fc(x.view(x.size(0),-1))
class PeDaMLP(nn.Module):
    """Parametric branch over the (Pe, Da) conditions."""
    def __init__(s, in_dim=2, out_dim=128, hidden=128, layers=3):
        super().__init__(); m=[nn.Linear(in_dim,hidden), nn.SiLU()]
        for _ in range(layers-2): m+=[nn.Linear(hidden,hidden), nn.SiLU()]
        m+=[nn.Linear(hidden,out_dim)]; s.net=nn.Sequential(*m)
    def forward(s, x): return s.net(x)
def make_trunk(in_dim, out_dim=128, layers=8, width=128):
    m=[nn.Linear(in_dim,width), nn.SiLU()]
    for _ in range(layers-2): m+=[nn.Linear(width,width), nn.SiLU()]
    m+=[nn.Linear(width,out_dim)]; return nn.Sequential(*m)
class PRTDeepONet(nn.Module):
    """branch1=[mask,ux,uy] (CNN) | branch2=[Pe,Da] (MLP) | trunk=[x,y,t,GDF] -> concentration(t)."""
    def __init__(s, nx=64, ny=148, trunk_in_dim=4, out_dim=128, branch1_ch=3, branch2_in_dim=2, cnn_blocks=5):
        super().__init__(); s.branch1_net=CustomCNN(branch1_ch,out_dim,num_blocks=cnn_blocks)
        s.branch2_net=PeDaMLP(branch2_in_dim,out_dim); s.trunk_net=make_trunk(trunk_in_dim,out_dim)
        s.bias=nn.Parameter(torch.zeros(1)); s.nx=nx; s.ny=ny
    def forward(s, b1, b2, tr):
        N,Lp,D=tr.shape; to=s.trunk_net(tr).unsqueeze(1)
        b1o=s.branch1_net(b1).unsqueeze(1).unsqueeze(2); b2o=s.branch2_net(b2).unsqueeze(1).unsqueeze(2)
        return ((b1o*b2o*to).sum(-1)+s.bias).view(N,s.nx,s.ny,1)


In [ ]:
# ====== 3. Data Loader ======
# The pre-built trunk4 datasets already hold branch1=mask, branch2=(Pe,Da), trunk=[x,y,t,GDF]
# and the transient labels y. Here we load them, then append the velocity-conditioning channels.
def load_dataset():
    train_ds=torch.load(f'{CACHE}/monod_train_dataset_trunk4.pt', weights_only=False)
    test_ds =torch.load(f'{CACHE}/monod_test_dataset_trunk4.pt',  weights_only=False)
    train_meta=list(torch.load(f'{CACHE}/monod_train_meta.pt', weights_only=False))
    test_meta =list(torch.load(f'{CACHE}/monod_test_meta.pt',  weights_only=False))
    b1_tr,b2_tr,tr_tr,y_tr=train_ds.tensors               # b1_tr: (N,1,nx,ny) = mask
    b1_te,b2_te,tr_te,y_te=test_ds.tensors                # meta[i] = (dom, group, time-snapshot)

    # --- masks per domain ---
    all_doms=sorted({m[0] for m in train_meta}|{m[0] for m in test_meta})
    MASK={}
    for dom in all_doms:
        with np.load(f'{MONNPZ}/m{dom}.npz') as d: MASK[dom]=(d['m'].reshape(nx,ny)>0.5).astype(int)

    # --- predicted velocity (ux,uy), Pe -> velocity group by mean Re ---
    VC=np.load(VELC_PATH); VELD={k:VC[k] for k in VC.files if k!='GROUP_MEANRE'}
    gm=VC['GROUP_MEANRE']; _ord=list(np.argsort(gm)); Pe2vg={1:int(_ord[0]),10:int(_ord[2])}
    def velfield(dom,Pe): return VELD[str(Pe2vg[Pe]*3000+dom)]   # (2,nx,ny) = [ux, uy]

    # --- z-score stats over train pore cells (dedup by (dom,group)) ---
    seen=set(); uvals=[]; vvals=[]
    for (dom,g,t) in train_meta:
        if (dom,g) in seen: continue
        seen.add((dom,g)); f=velfield(dom,pe_of(g)); mk=MASK[dom]==1
        uvals.append(f[0][mk]); vvals.append(f[1][mk])
    uvals=np.concatenate(uvals); vvals=np.concatenate(vvals)
    UMU,USD=float(uvals.mean()),float(uvals.std()); VMU,VSD=float(vvals.mean()),float(vvals.std())
    def u_z(dom,Pe): return ((velfield(dom,Pe)[0]-UMU)/USD).astype(np.float32)
    def v_z(dom,Pe): return ((velfield(dom,Pe)[1]-VMU)/VSD).astype(np.float32)

    # --- assemble branch1 = [mask, ux, uy] ---
    def vel_channel(meta):
        N=len(meta); ch=np.empty((N,2,nx,ny),np.float32); cc={}
        for i,(dom,g,t) in enumerate(meta):
            if (dom,g) not in cc: cc[(dom,g)]=(u_z(dom,pe_of(g)),v_z(dom,pe_of(g)))
            ch[i,0]=cc[(dom,g)][0]; ch[i,1]=cc[(dom,g)][1]
        return torch.from_numpy(ch)
    b1_tr2=torch.cat([b1_tr, vel_channel(train_meta)],dim=1)   # [mask, ux, uy]
    b1_te2=torch.cat([b1_te, vel_channel(test_meta)],dim=1)
    stats=dict(UMU=UMU,USD=USD,VMU=VMU,VSD=VSD,Pe2vg=Pe2vg)
    train_out=TensorDataset(b1_tr2,b2_tr,tr_tr,y_tr)
    test_out =TensorDataset(b1_te2,b2_te,tr_te,y_te)
    return train_out, test_out, (MASK,test_meta,stats)


In [ ]:
# ====== 4. Training Utilities ======
def train_model(model, train_dataset, test_dataset, num_epochs=EPOCHS, lr=LR, batch_size=BATCH, patience=PATIENCE):
    tl=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    vl=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)
    model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=lr); crit=nn.HuberLoss(delta=1.0)
    scaler=torch.cuda.amp.GradScaler(); best=None; bestloss=1e9; bestep=0; noimp=0; t0=time.time()
    for ep in range(1,num_epochs+1):
        model.train()
        for bb in tl:
            b1,b2,tr,y=[x.to(device) for x in bb]; opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=crit(model(b1,b2,tr),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); tot=0.0; n=0
        with torch.no_grad():
            for bb in vl:
                b1,b2,tr,y=[x.to(device) for x in bb]
                with torch.cuda.amp.autocast(): tot+=float(crit(model(b1,b2,tr),y).item()); n+=1
        vloss=tot/n
        if vloss<bestloss-1e-5: bestloss=vloss; bestep=ep; noimp=0; best=copy.deepcopy(model.state_dict())
        else:
            noimp+=1
            if noimp>=patience: print(f'early stop @ep{ep}', flush=True); break
        print(f'ep{ep:3d} val{vloss:.6f} best{bestloss:.6f}@{bestep} {time.time()-t0:.0f}s', flush=True)
    model.load_state_dict(best); return bestep


In [ ]:
# ====== 5. Evaluation Example (transient concentration RMSE over the porous ROI) ======
@torch.no_grad()
def evaluate(model, test_dataset, aux, num_samples=5):
    MASK,test_meta,_=aux; model.eval()
    for i in range(min(num_samples,len(test_dataset))):
        b1,b2,tr,y=test_dataset[i]
        pr=model(b1[None].to(device),b2[None].to(device),tr[None].to(device)).cpu().numpy().squeeze()
        gt=y.numpy().squeeze(); m=MASK[test_meta[i][0]]
        rmse=float(np.sqrt(np.mean((pr[m==1]-gt[m==1])**2)))
        print(f'  sample {i} (dom {test_meta[i][0]}, t {test_meta[i][2]}): RMSE = {rmse:.4f}')


In [ ]:
# ====== 6. Main Entry ======
if __name__ == '__main__':
    # 1) Load pre-built transient datasets + append velocity conditioning
    train_dataset, test_dataset, aux = load_dataset()
    print('train/test:', len(train_dataset), len(test_dataset))
    # 2) Build model:  branch1=[mask,ux,uy] | branch2=[Pe,Da] | trunk=[x,y,t,GDF]
    np.random.seed(SEED); torch.manual_seed(SEED)
    if device.type=='cuda': torch.cuda.manual_seed_all(SEED)
    model = PRTDeepONet(trunk_in_dim=4, branch1_ch=3, branch2_in_dim=2, cnn_blocks=5).to(device)
    # 3) Train (dense Huber loss)
    bestep = train_model(model, train_dataset, test_dataset)
    # 4) Evaluate
    evaluate(model, test_dataset, aux)
    # 5) Save parameters -> ../parameters/Monod.pt
    os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
    torch.save(model.state_dict(), MODEL_OUT)
    print('[save]', MODEL_OUT, '| best epoch', bestep)
